# Train Vision Transformer (ViT) on Euclid-like dataset

This notebook trains `SpectralViT` on the complex noisy Euclid-like dataset.
The ViT's global attention mechanism is well-suited for capturing long-range
spectral correlations caused by source blending across large spatial separations.

## Motivation for ViT on noisy data
In the Euclid NISP slitless configuration, a single bright source can contaminate
fainter sources located many tens of pixels away (the 1st-order trace extends
~128 pixels in the miniature setup, proportionally more at full NISP scale).
A convolutional architecture (U-Net) has limited receptive field per layer;
the ViT's self-attention spans the entire spectrogram image in a single operation,
giving it direct access to all blended contributions simultaneously.

## Training tips for noisy data
- Use **per-sample normalisation** to cope with the ~4 dex dynamic range between
  bright and faint sources after Poisson noise.
- Use **gradient clipping** (max norm=1.0) to stabilise training with noisy targets.
- The **spectral smoothness loss** (λ_spec=0.1) is especially important here
  because Poisson noise can cause the network to produce spiky spectra.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from spectangle.data_loaders.dataset import SpectangleDataModule
from spectangle.models.vit import SpectralViT
from spectangle.models.losses import CombinedLoss
from spectangle.utils.metrics import cube_metrics
from spectangle.utils.training import get_device

# ── Config ─────────────────────────────────────────────────────────────────
import spectangle as _spectangle_mod
PROJECT_ROOT  = Path(_spectangle_mod.__file__).parent.parent
H5_PATH       = PROJECT_ROOT / 'data' / 'raw' / 'complex_euclid_200s.h5'
IN_CHANNELS = 5
N_LAMBDA    = 128
PATCH_SIZE  = 8
EMBED_DIM   = 192      # larger than mini to handle noisy inputs
DEPTH       = 6
N_HEADS     = 6
LR          = 3e-5
N_EPOCHS    = 8
BATCH_SIZE  = 4
DEVICE      = get_device()
print(f'Device: {DEVICE}')

## 1. Data loading

In [ ]:
dm = SpectangleDataModule(
    h5_path=H5_PATH,
    batch_size=BATCH_SIZE,
    n_channels=IN_CHANNELS,
    normalise='per_sample',
    num_workers=0,
    seed=42,
)
print(dm)

scene_shape       = dm.scene_shape
spectrogram_shape = dm.spectrogram_shape

# Round up to next patch grid multiple
H_spec = ((spectrogram_shape[0] + PATCH_SIZE - 1) // PATCH_SIZE) * PATCH_SIZE
W_spec = ((spectrogram_shape[1] + PATCH_SIZE - 1) // PATCH_SIZE) * PATCH_SIZE
print(f'ViT input (padded to patch grid): H={H_spec}, W={W_spec}')
print(f'Number of patches: {(H_spec // PATCH_SIZE) * (W_spec // PATCH_SIZE)}')

## 2. Build the ViT model

In [ ]:
model = SpectralViT(
    in_channels=IN_CHANNELS,
    image_size_h=H_spec,
    image_size_w=W_spec,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    n_heads=N_HEADS,
    n_lambda=N_LAMBDA,
    scene_shape=scene_shape,
).to(DEVICE)

print(f'Parameters: {model.parameter_count():,}')

x_dummy = torch.zeros(2, IN_CHANNELS, H_spec, W_spec, device=DEVICE)
y_dummy = model(x_dummy)
print(f'Output shape: {y_dummy.shape}  (expected: [2, {N_LAMBDA}, {scene_shape[0]}, {scene_shape[1]}])')

## 3. Training
We use a higher spectral smoothness weight (`spectral=0.15`) compared to the
noiseless case because Poisson-dominated spectra benefit from stronger smoothing.

In [ ]:
loss_fn   = CombinedLoss(weights={'mse': 1.0, 'ssim': 0.5, 'spectral': 0.15})
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

def pad_input(x):
    ph = H_spec - x.shape[-2]
    pw = W_spec - x.shape[-1]
    return F.pad(x, (0, pw, 0, ph)) if (ph > 0 or pw > 0) else x

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total = 0.0; n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = pad_input(x.to(DEVICE)), y.to(DEVICE)
            pred = model(x)
            loss, _ = loss_fn(pred, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * x.shape[0]; n += x.shape[0]
    return total / max(n, 1)

tl_hist, vl_hist = [], []
for ep in range(1, N_EPOCHS + 1):
    tl = run_epoch(dm.train_dataloader(), True)
    vl = run_epoch(dm.val_dataloader(),   False)
    scheduler.step()
    tl_hist.append(tl); vl_hist.append(vl)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {ep:3d}/{N_EPOCHS} | train={tl:.5f} | val={vl:.5f} | lr={lr_now:.2e}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tl_hist, label='train'); ax.plot(vl_hist, label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True)
ax.set_title('ViT training curves (Euclid-like complex)')
plt.tight_layout(); plt.show()

## 4. Reconstruction evaluation

In [ ]:
model.eval()
x_s, y_s = dm._val_ds[0]
x_in = pad_input(x_s.unsqueeze(0).to(DEVICE))
with torch.no_grad():
    pred = model(x_in).squeeze(0).cpu().numpy()
gt = y_s.numpy()

m = cube_metrics(pred, gt)
print('Reconstruction metrics (val[0]):')
for k, v in m.items():
    print(f'  {k}: {v:.4f}')

# Broadband
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
axs[0].imshow(gt.sum(0),   cmap='gray',  origin='lower'); axs[0].set_title('GT broadband')
axs[1].imshow(pred.sum(0), cmap='gray',  origin='lower'); axs[1].set_title('ViT prediction')
axs[2].imshow(np.abs(pred.sum(0)-gt.sum(0)), cmap='magma', origin='lower'); axs[2].set_title('|Residual|')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()

# Wavelength slices
wav = np.linspace(9250, 18500, N_LAMBDA)
lam_idx = np.linspace(0, N_LAMBDA-1, 6, dtype=int)
fig, axs = plt.subplots(2, 6, figsize=(15, 5))
for col, li in enumerate(lam_idx):
    axs[0, col].imshow(gt[li],   cmap='gray', origin='lower'); axs[0, col].set_title(f'λ={wav[li]:.0f}Å')
    axs[1, col].imshow(pred[li], cmap='gray', origin='lower')
    axs[0, col].axis('off'); axs[1, col].axis('off')
axs[0, 0].set_ylabel('GT',   fontsize=9); axs[1, 0].set_ylabel('Pred', fontsize=9)
plt.suptitle('ViT: wavelength slices (top=GT, bottom=pred)')
plt.tight_layout(); plt.show()

## 5. Noise robustness analysis
Compare reconstruction quality as a function of input SNR.  Low-SNR samples
(dominated by sky + read noise) are the hardest to disentangle.

In [ ]:
model.eval()
all_psnr = []
all_snr  = []

for i in range(len(dm._val_ds)):
    x_s, y_s = dm._val_ds[i]
    x_in = pad_input(x_s.unsqueeze(0).to(DEVICE))
    with torch.no_grad():
        p = model(x_in).squeeze(0).cpu().numpy()
    g = y_s.numpy()

    # Approximate input SNR as peak-flux / noise-floor of spectrograms
    spec_arr = x_s[:4].numpy()
    snr = spec_arr.max() / (np.std(spec_arr[:, :8, :8]) + 1e-8)
    all_snr.append(float(snr))
    all_psnr.append(float(cube_metrics(p, g)['psnr']))

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(all_snr, all_psnr, alpha=0.7, s=20, color='steelblue')
ax.set_xlabel('Approx. input SNR'); ax.set_ylabel('PSNR (dB)')
ax.set_title('ViT reconstruction quality vs input SNR (validation set)')
ax.grid(True)
plt.tight_layout(); plt.show()

print(f'Mean PSNR : {np.mean(all_psnr):.2f} dB')
print(f'Median PSNR: {np.median(all_psnr):.2f} dB')

## 6. Model comparison (U-Net vs ViT vs PINN)
Load results saved from other euclid-like training notebooks to build a
side-by-side comparison table.

In [ ]:
# Example: manually fill after running the other notebooks
# Replace these placeholder values with your actual results.

results_table = {
    'Model': ['U-Net (4ch)', 'U-Net (5ch)', 'ViT (5ch)',  'PINN (4ch)'],
    'PSNR':  [0.0,           0.0,           m['psnr'],    0.0       ],
    'SSIM':  [0.0,           0.0,           m['ssim'],    0.0       ],
    'SAM':   [0.0,           0.0,           m['sam'],     0.0       ],
    'RMSE':  [0.0,           0.0,           m['rmse'],    0.0       ],
}

print(f'{"Model":<15}  {"PSNR":>8}  {"SSIM":>8}  {"SAM":>8}  {"RMSE":>8}')
print('-' * 55)
for row in zip(*results_table.values()):
    name, psnr, ssim, sam, rmse = row
    print(f'{name:<15}  {psnr:>8.3f}  {ssim:>8.4f}  {sam:>8.4f}  {rmse:>8.5f}')

print('\nNote: fill in U-Net and PINN values from their respective notebooks.')

## Next steps
- Run full training: `python scripts/train.py --config configs/models/vit.yaml`
- Enable the PINN physics loss with the ViT backbone by passing
  `backbone=SpectralViT(...)` to `PINN(backbone=..., ...)`.
- Increase dataset size to 1000+ samples for reliable generalisation.
- Try `use_realistic_seds=true` in `configs/simulations/complex_euclid.yaml`
  for galaxy-template SEDs.